# Clinical Trial Designer — GRPO Training V2
**Single-step evaluation | Fixed reward discrimination | Unsloth + TRL**

Key fixes from V1:
- Single-step reward (no multi-step random noise)
- Observation-rich prompts with available_actions
- Larger reward spread: good action ~+3.0, bad action ~-2.5
- Auto-clears Unsloth cache to prevent `has_images` error

## 1. Install Dependencies

In [ ]:
%%capture
# Pin trl==0.12.0 to avoid has_images NameError in Unsloth compiled cache
!pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
!pip install -q 'trl==0.12.0' requests scipy datasets matplotlib huggingface_hub

## 2. Configuration

In [ ]:
import os, json, csv, random, re, shutil
from datetime import datetime, timezone
import requests

ENV_URL   = "https://roopalgn-openenv-clinical-trial.hf.space"
DRY_RUN   = False     # True = validate pipeline only, no model
MODEL_SIZE = "1.5b"   # "1.5b" | "3b" | "7b"
EPISODES   = 20
SEED       = 42
NUM_GEN    = 8        # GRPO completions per prompt
ARTIFACT_DIR = "outputs/grpo_v2"

MODEL_PRESETS = {
    "1.5b": {"model_name": "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit", "lora_r": 8,  "seq_len": 2048},
    "3b":   {"model_name": "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",   "lora_r": 16, "seq_len": 3072},
    "7b":   {"model_name": "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",   "lora_r": 32, "seq_len": 4096},
}
preset = MODEL_PRESETS[MODEL_SIZE]

# Clear stale Unsloth compiled cache -> prevents has_images NameError
for d in ["unsloth_compiled_cache", "/tmp/unsloth_compiled_cache"]:
    if os.path.exists(d):
        shutil.rmtree(d)
        print(f"Cleared cache: {d}")

# Test connection
try:
    r = requests.get(f"{ENV_URL}/ping", timeout=10)
    print("Ping:", r.json())
except Exception as e:
    print(f"ERROR: Cannot connect: {e}")

print(f"Config: model={MODEL_SIZE}, episodes={EPISODES}, dry_run={DRY_RUN}")

## 3. Environment Functions + Single-Step Reward

In [ ]:
SYSTEM_PROMPT = """You are a clinical trial designer.
Choose the BEST next action from the available actions list.
Return exactly ONE valid JSON object — no extra text:
{"action_type": "<from available_actions>", "parameters": {}, "justification": "why", "confidence": 0.8}"""


def env_reset(seed=None):
    payload = {"seed": seed} if seed is not None else {}
    r = requests.post(f"{ENV_URL}/reset", json=payload, timeout=30)
    r.raise_for_status()
    return r.json()


def env_step(action_type, parameters=None, justification="", confidence=0.8):
    payload = {"action_type": action_type, "parameters": parameters or {},
               "justification": justification, "confidence": confidence}
    r = requests.post(f"{ENV_URL}/step", json=payload, timeout=30)
    r.raise_for_status()
    return r.json()


def extract_reward(result):
    reward = result.get("reward", 0.0)
    if isinstance(reward, dict):
        return float(sum(float(v) for v in reward.values()))
    return float(reward)


def parse_action(text, available_actions=None):
    """Extract JSON action from model output, validate against available_actions."""
    for src in re.findall(r"```json\s*(.*?)\s*```", text, re.DOTALL | re.IGNORECASE) + [text]:
        s, e = src.find("{"), src.rfind("}")
        if s == -1 or e <= s:
            continue
        try:
            parsed = json.loads(src[s:e+1])
            at = str(parsed.get("action_type", "")).strip()
            params = parsed.get("parameters", {})
            if not isinstance(params, dict): params = {}
            if available_actions and at not in available_actions:
                at = available_actions[0]
            return {"action_type": at or "set_primary_endpoint", "parameters": params}
        except Exception:
            continue
    return {"action_type": (available_actions[0] if available_actions else "set_primary_endpoint"), "parameters": {}}


def build_prompt(obs):
    """Observation-rich prompt so model knows what to choose."""
    available = obs.get("available_actions", ["set_primary_endpoint"])
    phase = obs.get("phase_data", {}).get("current_phase", "unknown")
    return (
        f"Scenario: {obs.get('scenario_description', '')}\n"
        f"Phase: {phase}\n"
        f"Resources: {json.dumps(obs.get('resource_status', {}))}\n"
        f"Available actions: {available}\n"
        f"Steps: {obs.get('steps_taken',0)}/{obs.get('max_steps',100)}\n"
        f"Hint: {obs.get('hint','')}\n\n"
        f"Choose ONE action from {available}.\n"
        'Return ONLY JSON: {"action_type": "<from list>", "parameters": {}, "justification": "...", "confidence": 0.8}'
    )


# Rolling seed counter — each GRPO completion gets a fresh env episode
_seed_ctr = [SEED + 10000]


def run_episode(model_response):
    """SINGLE-STEP reward — score ONE action. No multi-step noise."""
    try:
        seed = _seed_ctr[0]; _seed_ctr[0] += 1
        obs = env_reset(seed=seed)
        available = obs.get("available_actions", ["set_primary_endpoint"])
        action = parse_action(model_response, available)
        result = env_step(action["action_type"], action.get("parameters", {}))
        return extract_reward(result)
    except Exception as ex:
        print(f"Episode error: {ex}")
        return -2.0


def reward_func(completions, **kwargs):
    """GRPO reward function."""
    rewards = []
    for c in completions:
        text = c[-1]["content"] if isinstance(c, list) else str(c)
        rewards.append(run_episode(text))
    return rewards


print("Env functions ready. Single-step reward defined.")

## 4. Dry-Run Validation (No GPU)\n\nRun this to confirm reward discrimination BEFORE spending GPU time.

In [ ]:
if DRY_RUN:
    print("=== DRY RUN: Testing reward discrimination ===")
    deltas = []
    for ep in range(5):
        seed = SEED + ep
        obs = env_reset(seed=seed)
        available = obs.get("available_actions", ["set_primary_endpoint"])

        # Valid action (first available)
        r_valid = extract_reward(env_step(available[0]))

        # Invalid action (try submitting before it's allowed)
        env_reset(seed=seed)
        r_invalid = extract_reward(env_step("synthesize_conclusion"))

        delta = r_valid - r_invalid
        deltas.append(delta)
        print(f"  Ep {ep+1}: valid={r_valid:.3f}  invalid={r_invalid:.3f}  delta={delta:+.3f}")

    avg = sum(deltas)/len(deltas)
    print(f"\nAvg delta: {avg:.3f}")
    if avg > 0.5:
        print("✓ Rewards are discriminative — ready for training!")
    else:
        print("✗ Discrimination too low — check server deployment")
else:
    print("DRY_RUN=False — skipping validation cell.")

## 5. Load Model

In [ ]:
if not DRY_RUN:
    from unsloth import FastLanguageModel

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=preset["model_name"],
        max_seq_length=preset["seq_len"],
        dtype=None,
        load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=preset["lora_r"],
        lora_alpha=preset["lora_r"] * 2,
        lora_dropout=0.05,
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        bias="none",
        use_gradient_checkpointing="unsloth",
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    print(f"Model: {preset['model_name']}")
    model.print_trainable_parameters()
else:
    print("DRY_RUN=True — skipping model load.")

## 6. Build Training Dataset

In [ ]:
if not DRY_RUN:
    from datasets import Dataset

    print(f"Generating {EPISODES} diverse prompts...")
    chat_prompts = []
    for i in range(EPISODES):
        try:
            obs = env_reset(seed=SEED + i)
            user_content = build_prompt(obs)
        except Exception:
            user_content = "Design the next step of a clinical trial. Return JSON: {\"action_type\": \"set_primary_endpoint\", \"parameters\": {}}"
        chat_prompts.append({"prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_content},
        ]})

    train_dataset = Dataset.from_list(chat_prompts)
    print(f"Dataset: {len(train_dataset)} prompts from {EPISODES} unique seeds")
else:
    print("DRY_RUN=True — skipping dataset.")

## 7. Configure and Run GRPO Training

In [ ]:
if not DRY_RUN:
    import torch
    from trl import GRPOConfig, GRPOTrainer

    use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    use_fp16 = torch.cuda.is_available() and not use_bf16

    training_args = GRPOConfig(
        output_dir="checkpoints/grpo_v2",
        num_generations=NUM_GEN,
        max_completion_length=256,
        temperature=0.7,
        learning_rate=5e-6,
        num_train_epochs=1,
        per_device_train_batch_size=NUM_GEN,
        gradient_accumulation_steps=1,
        max_steps=EPISODES,
        warmup_steps=2,
        weight_decay=0.01,
        max_grad_norm=1.0,
        logging_steps=1,
        save_steps=max(1, EPISODES // 4),
        report_to="none",
        bf16=use_bf16,
        fp16=use_fp16,
    )

    trainer = GRPOTrainer(
        model=model,
        args=training_args,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        reward_funcs=[reward_func],
    )

    print(f"bf16={use_bf16}, fp16={use_fp16}")
    print(f"GRPO: {EPISODES} steps, {NUM_GEN} completions/step")
    print("Starting training...")
    start = datetime.now(timezone.utc)
    trainer.train()
    end = datetime.now(timezone.utc)
    print(f"Training done in {(end-start).total_seconds():.0f}s")
else:
    print("DRY_RUN=True — skipping training.")

## 8. Save Artifacts + Plot

In [ ]:
if not DRY_RUN:
    import numpy as np
    import matplotlib.pyplot as plt

    os.makedirs(ARTIFACT_DIR, exist_ok=True)
    reward_rows = [
        {"step": int(l.get("step", i)), "reward": float(l.get("reward", 0)),
         "reward_std": float(l.get("reward_std", 0)), "loss": float(l.get("loss", 0) or 0)}
        for i, l in enumerate(trainer.state.log_history, 1) if "reward" in l
    ]

    if reward_rows:
        rewards = [r["reward"] for r in reward_rows]
        steps   = [r["step"]   for r in reward_rows]

        # Save CSV
        csv_path = f"{ARTIFACT_DIR}/reward_log.csv"
        with open(csv_path, "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=["step","reward","reward_std","loss"])
            w.writeheader(); w.writerows(reward_rows)

        # Compute slope
        slope = float(np.polyfit(range(len(rewards)), rewards, 1)[0])

        # Save summary
        summary = {
            "model_size": MODEL_SIZE, "episodes": EPISODES,
            "mean_reward": float(np.mean(rewards)),
            "final_reward": float(rewards[-1]),
            "max_reward": float(max(rewards)),
            "min_reward": float(min(rewards)),
            "slope": slope,
            "runtime_seconds": (end-start).total_seconds(),
            "completed_at": end.isoformat(),
        }
        with open(f"{ARTIFACT_DIR}/training_summary.json", "w") as f:
            json.dump(summary, f, indent=2)

        # Plot
        trend = np.poly1d(np.polyfit(range(len(rewards)), rewards, 1))
        window = max(3, len(rewards)//5)
        rolling = np.convolve(rewards, np.ones(window)/window, mode="valid")

        fig, ax = plt.subplots(figsize=(10, 5))
        ax.scatter(steps, rewards, alpha=0.4, s=25, color="#4a90d9", label="Per-step reward")
        ax.plot(steps[window-1:], rolling, color="#e63946", lw=2, label=f"Rolling avg (w={window})")
        ax.plot(steps, trend(range(len(rewards))), "--", color="#2a9d8f", lw=1.5,
                label=f"Trend (slope={slope:+.4f})")
        ax.set_xlabel("Training step"); ax.set_ylabel("Reward")
        ax.set_title("GRPO Training Reward — V2 (Single-Step)")
        ax.legend(); ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(f"{ARTIFACT_DIR}/reward_curve.png", dpi=150)
        plt.show()

        print(f"mean={summary['mean_reward']:.3f}  slope={slope:+.4f}")
        print(f"Artifacts saved to {ARTIFACT_DIR}/")
    else:
        print("No reward rows captured.")
else:
    print("DRY_RUN=True — skipping artifacts.")

## 9. Push Checkpoint to HuggingFace Hub

In [ ]:
if not DRY_RUN:
    from huggingface_hub import login
    try:
        from google.colab import userdata
        login(token=userdata.get("HF_TOKEN"))
        print("Logged in via Colab secret.")
    except Exception:
        from huggingface_hub import notebook_login
        notebook_login()

    REPO_ID = "Roopalgn/clinical-trial-designer-grpo"
    model.push_to_hub(REPO_ID, commit_message=f"GRPO V2 {MODEL_SIZE}")
    tokenizer.push_to_hub(REPO_ID)
    print(f"Pushed to: https://huggingface.co/{REPO_ID}")
else:
    print("DRY_RUN=True — skipping push.")